In [1]:
import sys
from pathlib import Path

# --- find project root (where config.py lives) ---
CURRENT_PATH = Path().resolve()

PROJECT_ROOT = None
for parent in [CURRENT_PATH] + list(CURRENT_PATH.parents):
    if (parent / "config.py").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    raise RuntimeError("❌ config.py not found in any parent directory")

sys.path.append(str(PROJECT_ROOT))

from config import DATA_DIR, PROCESSED_DATA_DIR

print(f"✅ Project root set to: {PROJECT_ROOT}")


✅ Project root set to: C:\Users\ASUS\Desktop\FinalProject\protest_behavior_system


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# --- project root discovery ---
PROJECT_ROOT = Path().resolve()
sys.path.append(str(PROJECT_ROOT))

from config import DATA_DIR, PROCESSED_DATA_DIR


In [3]:
events_path = PROCESSED_DATA_DIR / "events_clean.csv"
df = pd.read_csv(events_path)

df.head()


,Protest ID #,Country name,COW Country Code,Year,Region,Protest,what number protest is this for this country year?,Start Day,Start Month,Start Year,...,Primary State Response to protests [Response 2?¾],Primary State Response to protests [Response 3],Primary State Response to protests [Response 4],Primary State Response to protests [Response 5],Primary State Response to protests [Response 6],Primary State Response to protests [Response 7],Notes,Demand_Type,RSI,Actor_Type
0,201990001,Canada,20,1990,North America,1,1,15.0,1.0,1990.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"""Canada's railway passenger system was finally...",political_demand,0,other
1,201990002,Canada,20,1990,North America,1,2,25.0,6.0,1990.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"protestors were only identified as ""young peop...",political_demand,0,other
2,201990003,Canada,20,1990,North America,1,3,1.0,7.0,1990.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"""THE Queen, after calling on Canadians to rema...",political_demand,0,other
3,201990004,Canada,20,1990,North America,1,4,12.0,7.0,1990.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"""Canada's federal government has agreed to acq...",land farm issue,1,other
4,201990005,Canada,20,1990,North America,1,5,14.0,8.0,1990.0,...,arrests,accomodation,NaN,NaN,NaN,NaN,Protests were directed against the state due t...,political_demand,3,other


In [5]:
event_counts = (
    df
    .groupby(["Country name", "Year"])
    .size()
    .reset_index(name="num_events")
)


In [8]:
required_cols = ["Country name", "Year"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["Year"] = df["Year"].astype(int)


In [12]:
event_counts = (
    df
    .groupby(["Country name", "Year"])
    .size()
    .reset_index(name="num_events")
)


In [13]:
group_cols = [
    "has_student_group",
    "has_political_group",
    "has_labor_group",
    "has_women_group"
]

available = [c for c in group_cols if c in df.columns]

group_memory = (
    df
    .groupby(["Country name", "Year"])[available]
    .mean()
    .reset_index()
)


In [17]:
if "protester_violence" in df.columns:
    violence_memory = (
        df
        .groupby(["Country name", "Year"])["protester_violence"]
        .mean()
        .reset_index(name="mean_protester_violence")
    )
else:
    violence_memory = None


In [18]:
memory = event_counts.merge(group_memory, on=["Country name", "Year"], how="left")

if violence_memory is not None:
    memory = memory.merge(violence_memory, on=["Country name", "Year"], how="left")

memory = memory.sort_values(["Country name", "Year"])
memory.head()


,Country name,Year,num_events
0,Afghanistan,1990,1
1,Afghanistan,1991,1
2,Afghanistan,1992,1
3,Afghanistan,1993,1
4,Afghanistan,1994,1


In [22]:
ROLLING_WINDOW = 5

memory_features = [
    c for c in memory.columns
    if c not in ["Country name", "Year"]
]

for col in memory_features:
    memory[f"{col}_hist"] = (
        memory
        .groupby("Country name")[col]
        .shift(1)
        .rolling(ROLLING_WINDOW, min_periods=1)
        .mean()
    )


In [26]:
final_cols = (
    ["Country name", "Year"] +
    [c for c in memory.columns if c.endswith("_hist")]
)

country_memory = memory[final_cols].dropna()

country_memory.head()


,Country name,Year,num_events_hist
1,Afghanistan,1991,1.0
2,Afghanistan,1992,1.0
3,Afghanistan,1993,1.0
4,Afghanistan,1994,1.0
5,Afghanistan,1995,1.0


In [30]:
output_path = PROCESSED_DATA_DIR / "country_memory.csv"
country_memory.to_csv(output_path, index=False)

print(f"✅ Country memory saved to {output_path}")


✅ Country memory saved to C:\Users\ASUS\Desktop\FinalProject\protest_behavior_system\data\processed\country_memory.csv
